# Week 6: the Fourier transform of images

Goals: magnitude and phase, non-locality of the Fourier transform, band coding.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import fourier, images

In [ ]:
camera = images.load_test_image("camera")
coins = images.load_test_image("coins")[:256, :256]
magnitude, phase = fourier.magnitude_phase(camera)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
images.show_image(camera, "image", ax=axes[0])
images.show_image(
    fourier.log_magnitude(magnitude), "log magnitude", ax=axes[1], scaled=True
)
images.show_image(phase, "phase", ax=axes[2], scaled=True)
plt.show()

## The phase carries the structure

In [ ]:
camera_small = camera[::2, ::2]
mag_a, phase_a = fourier.magnitude_phase(camera_small)
mag_b, phase_b = fourier.magnitude_phase(coins)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
images.show_image(
    fourier.from_magnitude_phase(mag_b, phase_a),
    "|coins|, phase of camera",
    ax=axes[0],
    scaled=True,
)
images.show_image(
    fourier.from_magnitude_phase(mag_a, phase_b),
    "|camera|, phase of coins",
    ax=axes[1],
    scaled=True,
)
plt.show()

## Non-locality

A small local change modifies every Fourier coefficient.

In [ ]:
changed = camera.copy()
changed[100:110, 240:250] = 255.0
difference = np.abs(np.fft.fft2(changed) - np.fft.fft2(camera))
np.count_nonzero(difference > 1e-6) / difference.size

## Band coding with a square low-pass mask

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, half_width in zip(axes, (16, 56, 128), strict=True):
    reconstruction, kept = fourier.band_code(camera, half_width)
    stats = fourier.compression_stats(kept)
    title = f"kept {kept:.2%}, ratio {stats['ratio']:.1f}, {stats['bpp']:.2f} bpp"
    images.show_image(reconstruction, title, ax=ax)
plt.show()